**Before starting this practical session**:

1. **Change the runtime type on Google Colab**: `Runtime > Change runtime type > T4 GPU`
2. **Install the packages below**:

In [ ]:
! pip install lightning torchmetrics torchinfo pyconll

3. Run this code to suppress some messages and warnings that may be displayed.

In [ ]:
import logging
logging.getLogger("lightning").setLevel(logging.ERROR)
logging.getLogger("lightning.pytorch.utilities.rank_zero").setLevel(logging.WARNING)
logging.getLogger("lightning.pytorch.accelerators.cuda").setLevel(logging.WARNING)
logger = logging.getLogger("lightning")
logger.propagate = False

import warnings
warnings.filterwarnings("ignore", ".*does not have many workers.*")
warnings.filterwarnings("ignore", ".*Missing logger folder.*")
warnings.filterwarnings("ignore", "`isinstance\\(treespec, LeafSpec\\)` is deprecated.*")

# A grammar exercise with a recurrent neural network

In this notebook, we will work on the [noun verb](https://github.com/google-research-datasets/noun-verb) dataset.
This dataset contains natural English sentences where there can be some ambiguity about the role of a word, i.e., whether it is a noun or a verb.

Below are a few samples from this dataset illustrating the task. The word in bold is the word to classify, and the word in italics after the sentence indicates the correct answer:

> Certain insects can damage plumerias, such as mites, **flies**, or aphids. *NOUN*

> **Mark** which area you want to distress. *VERB*

For example, the word **flies** can correspond to the third person singular of the verb *fly* (e.g., *time flies by so fast*), but also to the plural of the noun *fly*.
Similarly, the word **mark** can correspond to the verb *mark* or the noun *mark*.

The task is therefore a binary classification since there are only two possible classes: *noun* or *verb*.

## (Down)loading the dataset

The dataset is already split into three sets: training, validation, and test.
The corresponding files (`train.conll`, `dev.conll`, and `test.conll`) are available on this [GitHub repository](https://github.com/google-research-datasets/noun-verb).
The `load_dataset` function loads (and downloads if needed) the three datasets.

In [ ]:
def load_dataset(path="data"):
    """Load the noun verb dataset.

    Parameters
    ----------
    path : str
        Path to the directory.

    Returns
    -------
    train : Conll
        Training set.

    validation : Conll
        Validation set.

    test : Conll
        Test set.
    """
    import os
    import pyconll
    from urllib.request import urlretrieve

    if not os.path.exists(path):
        os.makedirs(path)

    files = ("train.conll", "dev.conll", "test.conll")

    # Downloads the files if necessary
    for file in files:
        if not os.path.isfile(os.path.join(path, file)):
            url = f"https://raw.githubusercontent.com/google-research-datasets/noun-verb/master/{file}"
            urlretrieve(url, os.path.join(path, file))

    return (pyconll.load_from_file(os.path.join(path, file)) for file in files)

Simply run the function to (down)load the three datasets:

In [ ]:
train, validation, test = load_dataset()

Let's visualize the machine learning task.
The `print_sample()` function displays the sample of a dataset corresponding to the given index.

In [ ]:
def print_sample(dataset, idx):
    """Print a sample from a data set.

    Parameters
    ----------
    dataset : Conll
        Data set.

    idx : int
        Index of the sample to print.
    """
    import re

    if not 0 <= idx < len(dataset):
        raise ValueError("Invalid index.")

    label = None
    index = None

    for i, token in enumerate(dataset[idx]):
        if len(token.feats):
            if label is not None:
                raise ValueError("Invalid sample. Try another index.")
            label = next(iter(token.feats["POS"]))
            index = i

    if label is None:
        raise ValueError("Invalid sample. Try another index.")

    sentence = " ".join([token.form for token in dataset[idx]])

    # Punctuation formatting
    for char in ",:;.?!)":
        pattern = f" *\\{char}" if char in ".?)" else f" *{char}"
        sentence = re.sub(pattern, char, sentence)

    for char in "(`":
        pattern = f"\\{char} *" if char in "(" else f"{char} *"
        sentence = re.sub(pattern, char, sentence)

    res = "Sentence" + "\n" + "=" * 8 + "\n"
    res += sentence
    res += "\n\n" + "Word" + "\n" + "=" * 4 + "\n"
    res += dataset[idx][index].form + f" (at position {index})"
    res += "\n\n" + "Label" + "\n" + "=" * 5 + "\n"
    res += label

    print(res)

### Question 1

Run this function to display a few samples from the training, validation, and test sets.

In [ ]:
# TODO

## Word embeddings

The task to perform falls under **natural language processing** since the input data are (English) sentences.
A sentence is a sequence of **tokens**.
A token is a unit of a language.
For example, words are tokens, but punctuation marks (comma, exclamation point, etc.) are also tokens.
However, it is not practical to work directly with tokens: machine learning models are mathematical functions whose inputs are *numerical*.
It is nevertheless possible to **transform tokens into numerical vectors**.
Such representations are known as *embeddings*.

A trivial approach would be to transform each word into a vector containing only zeros except for one element with a value of 1.
This approach is known in machine learning as **one-hot encoding** and can be used to transform a categorical variable into several binary variables.
However, this approach has many drawbacks:

* There are many different words in a language, for example hundreds of thousands in English and French.
* The data has been transformed, but no relevant information has been extracted in this representation.

A more useful approach is to **transform a word into a small, dense numerical vector**.
Here, we will not learn these representations ourselves, but we will use publicly available representations that have been learned on very large datasets.

A collection of such representations is known as [GloVe](https://nlp.stanford.edu/projects/glove/) (for *Global Vectors for Word Representation*).
There are several versions of these representations depending on the datasets on which these models were trained and the size of the representations.
We will use the first version of GloVe, which was trained on Wikipedia and news articles.
There are four versions for the representations with different dimensions: $50$, $100$, $200$, and $300$.

The command below downloads the archive containing the representations in the four different dimension sizes.

In [ ]:
! mkdir -p data && cd data && (test -f glove.6B.zip || wget https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip) && unzip -o -q glove.6B.zip

We will use the smallest representations, i.e., dimension $50$. **In summary, we will transform each token into a dense numerical vector of dimension 50**.

For tokens that are not part of the GloVe vocabulary, we will create a specific token, called `out_of_vocabulary`, which we will compute as the average vector of all representations.

The `load_embeddings()` function loads the data into a dictionary and computes the average representation for the `out_of_vocabulary` token.
Each key of the dictionary is a token and its value is the vector representation of that token.

In [ ]:
import numpy as np
import torch


def load_embeddings(path="data"):
    """Load the GloVe 50d embeddings.

    Parameters
    ----------
    path : str
        Path to the directory

    Returns
    -------
    embeddings : dict
        GloVe 50d embeddings.
    """
    import os

    # Save the results in a dictionary
    embeddings = {}

    # Get the embedding for each token in the file
    mean = torch.zeros(50, dtype=torch.float32)
    with open(os.path.join(path, "glove.6B.50d.txt"), "r") as file:
        for line in file.readlines():
            split = line.strip().split(" ")
            token = split[0]
            value = torch.from_numpy(np.array(split[1:], dtype=np.float32))
            embeddings[token] = value
            mean += value

    # Define the mean embedding for out-of-vocabulary token
    embeddings["out_of_vocabulary"] = mean / len(embeddings)

    return embeddings

We run this function to (down)load these English word embeddings.

In [ ]:
embeddings = load_embeddings()

### Question 2

Display the vector representations of a few common English words.

> **Remark**: All the words in the dictionary are in lowercase.

In [ ]:
# TODO

### Question 3

Learned vector representations are known to have certain linear structures.

* Compute the vector representation `king + (woman - man)` and compare it to the vector representation of `queen`.
* Compute the vector representation `strong + (lighter - light)` and compare it to the vector representation of `stronger`.

You can use the *cosine similarity* function to determine the similarity between two vectors. Cosine similarity is defined as:
$$
    \cos(x, y) = \frac{x^\top y}{\Vert x \Vert \Vert y \Vert}
$$
Its values lie in the interval $[-1, 1]$. The higher its value, the more similar the vectors are.
The `cosine_similarity()` function defined below computes the cosine similarity between two tensors.

In [ ]:
def cosine_similarity(x, y):
    """Computes the cosine similarity score between two tensors.

    Parameters
    ----------
    x, y : Tensor, shape = (embedding_size,)
        The two tensors to be compared.

    Returns
    -------
    float
        The cosine similarity score between both tensors.
    """
    return ((x @ y) / ((x @ x) * (y @ y)) ** 0.5).item()

In [ ]:
# TODO

## Data preprocessing

Now that we have vector representations for the tokens and a better understanding of the data, it is time to preprocess this data.
For each sample, we will:
* transform the sequence of tokens into a sequence of tensors,
* retrieve the label of the token to classify, and
* retrieve the index of the token to classify.

The `preprocess_dataset()` function performs this processing on the input dataset and returns these three variables.

In [ ]:
def preprocess_dataset(dataset, embeddings):
    """Preprocess a data set.

    Parameters
    ----------
    dataset : Conll
        Data set.

    embeddings : dict
        Embedding of each token.

    Returns
    -------
    x : list[Tensors]
        Preprocessed inputs.

    y : Tensor
        Preprocessed outputs.

    index : Tensor
        Indices of the tokens of interest.
    """

    x = []
    y = []
    index = []

    label_mapping = {"NON-VERB": 0, "VERB": 1}

    for sentence in dataset:

        embedded_sentence = []
        label = None
        idx = None

        for i, token in enumerate(sentence):

            # Get the token in lowercase
            token_lower = token.form.lower()

            # Get the embedding of the token
            if token_lower in embeddings.keys():
                embedded_sentence.append(embeddings[token_lower].reshape(1, -1))
            else:
                embedded_sentence.append(embeddings["out_of_vocabulary"].reshape(1, -1))

            # Get the label (if any)
            if len(token.feats):
                if label is not None:
                    raise ValueError("Two annotated tokens in a single sentence.")
                label = label_mapping[next(iter(token.feats["POS"]))]
                idx = i

        # Add the preprocessed sample to the dataset only if there is a label available
        if label is not None:
            x.append(torch.concat(embedded_sentence))
            y.append(label)
            index.append(idx)

    y = torch.tensor(y).to(dtype=torch.float32)
    index = torch.tensor(index).to(dtype=torch.int64)

    return x, y, index

We just need to run this function on the three datasets.

In [ ]:
x_train, y_train, index_train = preprocess_dataset(train, embeddings)
x_val, y_val, index_val = preprocess_dataset(validation, embeddings)
x_test, y_test, index_test = preprocess_dataset(test, embeddings)

### Question 4

The `y_train`, `y_val`, and `y_test` tensors are binary tensors (each element is $0$ or $1$). A $0$ corresponds to a noun, while a $1$ corresponds to a verb.
Compute the proportion of verbs on the three sets to get an idea of the accuracy, i.e., the proportion of correct predictions, of a trivial model.
You can use the [`torch.Tensor.mean()`](https://pytorch.org/docs/stable/generated/torch.Tensor.mean.html) method.

In [ ]:
# TODO

## Bidirectional gated recurrent unit

### Gated recurrent unit

The **gated recurrent unit** is a type of recurrent layer.
It is a simplified variant of *long short-term memory*:

* The **forget and input gates** are merged into a single **update gate**.
* The **memory cell** and the **hidden state** are merged into a single **hidden state**.

The image below summarizes the architecture of the gated recurrent unit:

[<img src="https://raw.githubusercontent.com/johannfaouzi/smart-data-deep-learning/main/figures/gru_full.png" width="500"/>](https://raw.githubusercontent.com/johannfaouzi/smart-data-deep-learning/main/figures/gru_full.png)

### Bidirectional recurrent layer

We want to classify a specific word in a given sentence, either as a noun or a verb.
To perform this task manually, a human would use the entire sentence to classify the word.
If we use the entire sequence of tokens in a classic recurrent layer, the model does not know which word it should classify in the sentence.
Conversely, if $k$ is the index of the token to classify and if we only use the first $k$ tokens of the sequence, the following tokens are not used.
To solve this problem, we will use a **bidirectional** recurrent layer.

A bidirectional recurrent layer considers a sequence in both directions:
* the forward direction: $ x^{(0)} \longrightarrow x^{(1)} \longrightarrow \ldots \longrightarrow x^{(t-1)} \longrightarrow x^{(t)} $
* the backward direction: $ x^{(0)} \longleftarrow x^{(1)} \longleftarrow \ldots \longleftarrow x^{(t-1)} \longleftarrow x^{(t)} $

The image below illustrates this principle:

[<img src="https://raw.githubusercontent.com/johannfaouzi/smart-data-deep-learning/main/figures/rnn_bidir.png" width="300"/>](https://raw.githubusercontent.com/johannfaouzi/smart-data-deep-learning/main/figures/rnn_bidir.png)


### Operation performed here

**If $k$ is the index of the token of interest, we will extract the tensors $\overrightarrow{h}^{(k)}$ (the hidden state for the token of interest in the forward direction) and $\overleftarrow{h}^{(k)}$ (the hidden state for the token of interest in the backward direction)**.
This way, we tell the model which word we want to classify while still using all the words of the sentence.

The image below illustrates the operation performed here:

[<img src="https://raw.githubusercontent.com/johannfaouzi/smart-data-deep-learning/main/figures/brnn_specific_token.png" width="450"/>](https://raw.githubusercontent.com/johannfaouzi/smart-data-deep-learning/main/figures/brnn_specific_token.png)


### A few words on variable-length sequences

A particularity of textual data is that the sequence length is not fixed: the number of words in a sentence varies from one sentence to another.
When working on a batch of sequences, we cannot directly create a new higher-dimensional tensor containing all the sequences because the sequences can have different lengths.

To address this problem, one possible solution is to apply **padding** to the shortest sequences.
This way, the (possibly padded) sequences are all the same length and can be processed efficiently in batches.
However, this is only a **computational trick**: we do not want to use the padding in the processing itself.
Therefore, we also need to indicate which values were padded so they are not processed.

PyTorch provides tools to implement this solution:

* [`torch.nn.utils.rnn.pad_sequence()`](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html): this function pads a list of variable-length tensors.
* [`torch.nn.utils.rnn.pack_padded_sequence()`](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html): this function packs a tensor containing padded variable-length sequences.
* [`torch.nn.utils.rnn.pad_packed_sequence()`](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_packed_sequence.html): this function pads a packed batch of variable-length sequences (the inverse operation of [`pack_padded_sequence()`](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html)).

Here is how to use these tools:

1. When dealing with variable-length sequences, the input of a recurrent layer must be an instance of [PackedSequence](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.PackedSequence.html). To obtain such an instance of this class, you must first pad the sequences with `pad_sequence()` and then pack these sequences with `pack_padded_sequence()`.
2. When the input of a recurrent layer is an instance of `PackedSequence`, its output is also an instance of `PackedSequence`. You simply need to unpack these sequences with `pad_packed_sequence()`.

Here is an illustration of how to use these tools:

```python
import torch
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

sequence1 = torch.rand(6, 50)  # a sequence of length 6
sequence2 = torch.rand(10, 50)  # a sequence of length 10
sequence3 = torch.rand(9, 50)  # a sequence of length 9

sequences = [sequence1, sequence2, sequence3]

# We compute the length of the sequences
lens = [sequence.size()[0] for sequence in sequences]

# We pad this list of sequences
sequences_padded = pad_sequence(sequences)

# We pack these possibly padded sequences
sequences_packed = pack_padded_sequence(sequences_padded, lens, enforce_sorted=False)

# We pad the packed sequences
sequences_unpacked, lens_unpacked = pad_packed_sequence(sequences_packed)
```

### Question 5

We will build a recurrent neural network with the following architecture:

* First layer: bidirectional GRU layer with 100 hidden variables.
* Second layer: linear layer with 1 output variable.

A few important notes:

* Since we have a bidirectional GRU, the hidden state is a vector of size $200 = 100 \times 2$ because the two tensors are automatically concatenated.
* We only want to retrieve the hidden state of the token we are interested in (i.e., the word to classify).
* For your implementation of the `forward()` method, you will assume that the input `x` is a list of tensors (one tensor per sentence) and that `index` is a tensor with the index of the token of interest in each sentence (we will see later how to obtain this result).

Complete the missing code in the `__init__()` and `forward()` methods of the `RecurrentNeuralNetwork()` class.

In [ ]:
import lightning as L
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torchmetrics import Accuracy


class RecurrentNeuralNetwork(L.LightningModule):  # Always inherit from the lightning.LightningModule class

    def __init__(self):
        """Constructor.
        
        In the constructor, we always run the constructor of the parent class and we
        instantiate all the elements of our network: the layers, the activation functions,
        the loss function and the metrics.
        """
        super().__init__()  # Always run the constructor of the parent class

        ### BEGIN TODO ###
        # self.gru =
        # self.linear =
        ### END TODO ###

        self.loss = torch.nn.BCEWithLogitsLoss()
        self.accuracy_train = Accuracy(task="binary")
        self.accuracy_val = Accuracy(task="binary")
        self.accuracy_test = Accuracy(task="binary")

    def forward(self, x, index):
        """Implements the forward pass.
        
        Parameters
        ----------
        x : list[Tensor], len = batch_size
            Batch of sequences of vectors.
        
        index : tensor, shape = (batch_size,)
            Index of the token of interest in each sequence.

        Returns
        -------
        logits : Tensor, shape = (batch_size,)
            Logits.
        """
        ### BEGIN TODO ###
        # Compute the length of each sequence in this batch

        # Pad the smallest sequences

        # Pack the padded sequences

        # Apply the GRU layer

        # Unpack the output of the GRU layer

        # Get the hidden state of the token of interest

        # Apply the linear layer
        # logits =
        #### END TODO ####

        return logits

    def step(self, batch, dataset):
        """Perform a step.
        
        A step consists in mapping a batch of samples to the loss function for this batch of input samples.

        Parameters
        ----------
        batch : tuple
            A batch of samples.
            
        dataset : {"training", "validation"}
            Data set used.

        Returns
        -------
        loss : Tensor, shape = (1,)
            Loss function evaluated for this batch of samples.
        """
        # Get the inputs, outputs, and indices for this batch
        x = [item[0].to(self.device) for item in batch]
        y = torch.tensor([item[1] for item in batch], device=self.device)
        index = torch.tensor([item[2] for item in batch], device=self.device)

        logits = self(x, index)  # Forward pass outputting the logits
        loss = self.loss(logits, y)  # Evaluation of the loss function
        y_pred = (logits > 0).to(torch.float32)  # Predictions of the model

        if dataset == "training":
            metric = self.accuracy_train
            name = "train"
            bar_step = True
        elif dataset == "validation":
            metric = self.accuracy_val
            name = "val"
            bar_step = False
        else:
            metric = self.accuracy_test
            name = "test"
            bar_step = False

        acc = metric(y_pred, y)
        self.log(f"loss_{name}", loss, batch_size=len(y), prog_bar=bar_step, on_step=bar_step, on_epoch=True)
        self.log(f"accuracy_{name}", acc, batch_size=len(y), prog_bar=bar_step, on_step=bar_step, on_epoch=True)

        return loss

    def training_step(self, batch):
        """Perform a training step."""
        return self.step(batch, "training")

    def validation_step(self, batch):
        """Perform a validation step."""
        return self.step(batch, "validation")

    def test_step(self, batch):
        """Perform a test step."""
        return self.step(batch, "test")

    def on_train_start(self):
        """Code to run at the beginning of training."""
        string = f"Version {self.trainer.logger.version}"
        print(f"{string}\n{'=' * len(string)}\n")

    def on_train_epoch_end(self):
        """Code to run at the end of each training epoch."""
        metrics = self.trainer.callback_metrics
        string = (f"""
            Epoch {self.trainer.current_epoch + 1} / {self.trainer.max_epochs}
            -----------------------------------------
            |  Data set  | Loss function | Accuracy |
            | ---------- | ------------- | -------- |
            |  Training  |{metrics['loss_train'].item():^15.5f}|{metrics['accuracy_train'].item():^10.4%}|
            | Validation |{metrics['loss_val'].item():^15.5f}|{metrics['accuracy_val'].item():^10.4%}|
            -----------------------------------------
        """)
        string = "\n".join([line.strip() for line in string.strip().split("\n")])
        print(string, "\n")

    def configure_optimizers(self):
        """Configure the optimization algorithms."""
        optimizer = torch.optim.Adam(self.parameters())
        return optimizer

## A few words about the `DataLoader`

We have seen that an instance of [`torch.utils.data.DataLoader()`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) retrieves batches of samples of size `batch_size`, with possible shuffling of samples at each epoch (if `shuffle=True`).
However, a `DataLoader` does much more under the hood.

In particular, with the default values of certain hyperparameters, it transforms a list of tensors into a single tensor. For example:

* If each input is a 1-dimensional tensor of size $50$ and `batch_size=6`, then the size of the returned tensor (corresponding to the batch of samples) is $(6, 50)$.
* If each input is a 2-dimensional tensor of size $(32, 32)$ and `batch_size=4`, then the size of the returned tensor (corresponding to the batch of samples) is $(4, 32, 32)$.

The returned tensor has an additional dimension (the first dimension) corresponding to the samples.

As we saw in the previous section, such a transformation is impossible with tensors of different lengths.
Therefore, it is necessary to disable this transformation and implement our own transformation (with padding and packing).

The `DataLoader` hyperparameter that controls this transformation is `collate_fn` (see the [documentation](https://pytorch.org/docs/stable/data.html#working-with-collate-fn) for this hyperparameter).
Its value must be `None` or a callable, so in practice a function.
We can disable the transformation by giving the identity function as the argument for this hyperparameter, i.e., `collate_fn=lambda x: x`.

By making this change, the `DataLoader` will now return a list of length `batch_size`, each element of this list being what is returned by the `__getitem__()` method of the instance of the custom class inheriting from `Dataset`.

> **Remark**: Another possibility would have been to perform the padding and packing steps in the `collate_fn` function.

### Question 6

Create your own `CustomDataset` class inheriting from the [`torch.utils.data.Dataset()`](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset) class. As a reminder, this class must define the following three methods:

* `__len__()`: this method returns the number of samples in the dataset.
* `__getitem__()`: this method loads and returns the $n$-th sample of the dataset (where $n$ is an argument of the method)
* `__init__()`: this method defines all the information needed to implement the other two methods.

Then create a `DataLoader` for each of the three sets (training, validation, test) with batches of size $64$ (`batch_size=64`).
Don't forget to shuffle the samples in the training set (`shuffle=True`).

In [ ]:
# TODO

We will now train our model for 10 epochs.

In [ ]:
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger


model = RecurrentNeuralNetwork()

trainer = L.Trainer(
    max_epochs=10,
    enable_model_summary=False,  # disable model summary
    logger=CSVLogger('.'),  # save the results in a CSV file
    num_sanity_val_steps=0,  # do not perform any validation step before training
    callbacks=[TQDMProgressBar(refresh_rate=100)]  # update the progress bar every 100 batches
)

trainer.fit(
    model=model,
    train_dataloaders=dataloader_train,
    val_dataloaders=dataloader_val
)

## Hyperparameter optimization

The class coded above is not very flexible, in the sense that if we want to change the values of certain hyperparameters, we must create a new class.
It is better to have a flexible class, with input arguments allowing us to change the values of these hyperparameters when creating an instance of this class.

The `RecurrentNeuralNetworkAdvanced` class defined below allows some customization of the model's architecture.
We still have a recurrent neural network with one or more recurrent layers followed by one or more linear layers.
However, the following modifications are possible thanks to the class arguments:

* We can choose the type of recurrent layer (RNN / LSTM / GRU).
* We can choose the number of recurrent layers.
* We can choose the number of variables in the hidden state of the recurrent layers.
* We can choose the number of linear layers as well as their sizes.
* We can choose the activation function to use between the linear layers.

In [ ]:
import lightning as L
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torchmetrics import Accuracy


class RecurrentNeuralNetworkAdvanced(L.LightningModule):  # Always inherit from the lightning.LightningModule class
    """General class for our recurrent neural networks.

    Parameters
    ----------
    rnn_type : {"rnn", "lstm", "gru"}
        Type of recurrent layer.
    """

    def __init__(
        self,
        rnn_type="gru",
        n_rnn=1,
        hidden_size_rnn=100,
        n_linear=1,
        output_size_linear=100,
        activation=torch.nn.ReLU()
    ):
        """Constructor.
        
        In the constructor, we always run the constructor of the parent class and we
        instantiate all the elements of our network: the layers, the activation functions,
        the loss function and the metrics.
        """
        super().__init__()  # Always run the constructor of the parent class

        # Recurrent layers
        if rnn_type == "rnn":
            self.rnn = torch.nn.RNN(
                50, hidden_size=hidden_size_rnn, num_layers=n_rnn, bidirectional=True
            )
        elif rnn_type == "lstm":
            self.rnn = torch.nn.LSTM(
                50, hidden_size=hidden_size_rnn, num_layers=n_rnn, bidirectional=True
            )
        elif rnn_type == "gru":
            self.rnn = torch.nn.GRU(
                50, hidden_size=hidden_size_rnn, num_layers=n_rnn, bidirectional=True
            )
        else:
            raise ValueError("rnn must be one of 'rnn', 'lstm' or 'gru'.")

        # Linear layers
        size_linear = [2 * hidden_size_rnn]

        if isinstance(output_size_linear, int):
            size_linear += [output_size_linear] * (n_linear - 1)
        else:
            size_linear += list(output_size_linear)
        size_linear += [1]

        self.linear = torch.nn.Sequential(
            *(
                [torch.nn.Sequential(torch.nn.Linear(in_features, out_features), activation)
                 for in_features, out_features in zip(size_linear[:-2], size_linear[1:-1])]
                + [torch.nn.Linear(size_linear[-2], size_linear[-1])]
            )
        )

        self.loss = torch.nn.BCEWithLogitsLoss()
        self.accuracy_train = Accuracy(task="binary")
        self.accuracy_val = Accuracy(task="binary")
        self.accuracy_test = Accuracy(task="binary")

    def forward(self, x, index):
        """Implements the forward pass.

        Parameters
        ----------
        x : list[Tensor], len = batch_size
            Batch of sequences of vectors.

        index : tensor, shape = (batch_size,)
            Index of the token of interest in each sequence.

        Returns
        -------
        logits : Tensor, shape = (batch_size,)
            Logits.
        """
        # Compute the length of each sequence in this batch
        lens = [sequence.size()[0] for sequence in x]

        # Pad the smallest sequences
        x_padded = pad_sequence(x)

        # Pack the padded sequences
        x_packed = pack_padded_sequence(x_padded, lens, enforce_sorted=False)

        # Apply the recurrent layer(s)
        output_packed, _ = self.rnn(x_packed)

        # Unpack the output of the recurrent layer(s)
        output_unpacked, _ = pad_packed_sequence(output_packed)

        # Get the hidden state of the token of interest
        h = torch.concat([output_unpacked[idx, i].reshape(1, -1) for i, idx in enumerate(index)])

        # Apply the linear layer(s)
        logits = torch.squeeze(self.linear(h), dim=1)

        return logits

    def step(self, batch, dataset):
        """Perform a step.
        
        A step consists in mapping a batch of samples to the loss function for this batch of input samples.

        Parameters
        ----------
        batch : tuple
            A batch of samples.
            
        dataset : {"training", "validation"}
            Data set used.

        Returns
        -------
        loss : Tensor, shape = (1,)
            Loss function evaluated for this batch of samples.
        """
        # Get the inputs, outputs, and indices for this batch
        x = [item[0].to(self.device) for item in batch]
        y = torch.tensor([item[1] for item in batch], device=self.device)
        index = torch.tensor([item[2] for item in batch], device=self.device)

        logits = self(x, index)  # Forward pass outputting the logits
        loss = self.loss(logits, y)  # Evaluation of the loss function
        y_pred = (logits > 0).to(torch.float32)  # Predictions of the model

        if dataset == "training":
            metric = self.accuracy_train
            name = "train"
            bar_step = True
        elif dataset == "validation":
            metric = self.accuracy_val
            name = "val"
            bar_step = False
        else:
            metric = self.accuracy_test
            name = "test"
            bar_step = False

        acc = metric(y_pred, y)
        self.log(f"loss_{name}", loss, batch_size=len(y), prog_bar=bar_step, on_step=bar_step, on_epoch=True)
        self.log(f"accuracy_{name}", acc, batch_size=len(y), prog_bar=bar_step, on_step=bar_step, on_epoch=True)

        return loss

    def training_step(self, batch):
        """Perform a training step."""
        return self.step(batch, "training")

    def validation_step(self, batch):
        """Perform a validation step."""
        return self.step(batch, "validation")

    def test_step(self, batch):
        """Perform a test step."""
        return self.step(batch, "test")

    def on_train_start(self):
        """Code to run at the beginning of training."""
        string = f"Version {self.trainer.logger.version}"
        print(f"{string}\n{'=' * len(string)}\n")

    def on_train_epoch_end(self):
        """Code to run at the end of each training epoch."""
        metrics = self.trainer.callback_metrics
        string = (f"""
            Epoch {self.trainer.current_epoch + 1} / {self.trainer.max_epochs}
            -----------------------------------------
            |  Data set  | Loss function | Accuracy |
            | ---------- | ------------- | -------- |
            |  Training  |{metrics['loss_train'].item():^15.5f}|{metrics['accuracy_train'].item():^10.4%}|
            | Validation |{metrics['loss_val'].item():^15.5f}|{metrics['accuracy_val'].item():^10.4%}|
            -----------------------------------------
        """)
        string = "\n".join([line.strip() for line in string.strip().split("\n")])
        print(string, "\n")

    def configure_optimizers(self):
        """Configure the optimization algorithms."""
        optimizer = torch.optim.Adam(self.parameters())
        return optimizer

### Question 7

Initialize a few models with different combinations for the values of these hyperparameters.
Do not train the models, but perform a forward pass on a small batch of samples to make sure no error is raised.

In [ ]:
# TODO

### Question 8

Use your new custom class to initialize a model with an architecture different from the first model.
**Remember that the deeper your model is and the more trainable parameters it has, the longer training will take.**
If you have enough time, try several architectures.
In any case, save all your models!

In [ ]:
# TODO

### Question 9

In theory, we should choose the best model on the validation set and evaluate it on the test set. Out of curiosity, we will here evaluate all models on the test set.

In [ ]:
# TODO

## Going further: using the characters of the words

Our models rely only on the GloVe embeddings of the tokens. This representation has several limitations for our task:

* **Out-of-vocabulary tokens** all share the same vector (`out_of_vocabulary`), so the model knows nothing about them.
* **The spelling of a word is ignored**, although it carries useful information: suffixes such as *-s*, *-ed*, *-ing*, *-tion* or *-ment* are strong clues for distinguishing nouns from verbs.
* **The case is lost**, since all the tokens are converted to lowercase. However, a capitalized word at the beginning of a sentence is often a verb in the imperative mood (e.g., *"**Mark** which area you want to distress."*), which is one of the most difficult cases of this dataset.

The authors of the dataset ([Elkahky et al., 2018](https://aclanthology.org/D18-1277/)) used a model that also builds a representation of each word from its characters.
We will follow the same idea and add two types of information to the GloVe embedding of each token:

* a **character-level representation** computed by a small 1D convolutional neural network applied to the sequence of characters of the token;
* $4$ **binary features**: the token starts with an uppercase letter, the token is entirely in uppercase, the token is the first one of the sentence, the token contains a digit.

### Data preprocessing

Each character is mapped to an integer: $0$ is reserved for padding, $1$ for the characters that are rare or absent from the training set, and the other characters get an integer starting from $2$.
Each token is truncated to its first $20$ characters and padded with $0$ if it is shorter.
The `build_char_mapping()` and `preprocess_dataset_char()` functions defined below perform these steps.

In [ ]:
def build_char_mapping(dataset, min_count=5):
    """Build the mapping between the characters and the integers.

    Parameters
    ----------
    dataset : Conll
        Data set (the training set).

    min_count : int (default = 5)
        Minimum number of occurrences for a character to be kept.

    Returns
    -------
    char_mapping : dict
        Integer associated with each character. 0 is reserved for padding,
        and 1 for unknown characters.
    """
    from collections import Counter

    counts = Counter(char for sentence in dataset for token in sentence for char in token.form)
    chars = sorted(char for char, count in counts.items() if count >= min_count)
    return {char: i + 2 for i, char in enumerate(chars)}


def preprocess_dataset_char(dataset, embeddings, char_mapping, max_chars=20):
    """Preprocess a data set with character-level information.

    Parameters
    ----------
    dataset : Conll
        Data set.

    embeddings : dict
        Embedding of each token.

    char_mapping : dict
        Integer associated with each character.

    max_chars : int (default = 20)
        Maximum number of characters kept for each token.

    Returns
    -------
    x : list[Tensor]
        GloVe embeddings of the tokens, shape = (n_tokens, 50) for each sentence.

    chars : list[Tensor]
        Characters of the tokens, shape = (n_tokens, max_chars) for each sentence.

    casing : list[Tensor]
        Binary features of the tokens, shape = (n_tokens, 4) for each sentence.

    y : Tensor
        Preprocessed outputs.

    index : Tensor
        Indices of the tokens of interest.
    """
    x = []
    chars = []
    casing = []
    y = []
    index = []

    label_mapping = {"NON-VERB": 0, "VERB": 1}

    for sentence in dataset:

        embedded_sentence = []
        chars_sentence = torch.zeros(len(sentence), max_chars, dtype=torch.int64)
        casing_sentence = torch.zeros(len(sentence), 4)
        label = None
        idx = None

        for i, token in enumerate(sentence):
            form = token.form

            # GloVe embedding of the token in lowercase
            embedded_sentence.append(embeddings.get(form.lower(), embeddings["out_of_vocabulary"]))

            # Characters of the token (1 = unknown character, 0 = padding)
            char_ids = [char_mapping.get(char, 1) for char in form[:max_chars]]
            chars_sentence[i, :len(char_ids)] = torch.tensor(char_ids)

            # Binary features
            casing_sentence[i] = torch.tensor([
                form[0].isupper(),                     # starts with an uppercase letter
                form.isupper() and len(form) > 1,      # entirely in uppercase
                i == 0,                                # first token of the sentence
                any(char.isdigit() for char in form),  # contains a digit
            ])

            # Get the label (if any)
            if len(token.feats):
                if label is not None:
                    raise ValueError("Two annotated tokens in a single sentence.")
                label = label_mapping[next(iter(token.feats["POS"]))]
                idx = i

        # Add the preprocessed sample to the dataset only if there is a label available
        if label is not None:
            x.append(torch.stack(embedded_sentence))
            chars.append(chars_sentence)
            casing.append(casing_sentence)
            y.append(label)
            index.append(idx)

    y = torch.tensor(y).to(dtype=torch.float32)
    index = torch.tensor(index).to(dtype=torch.int64)

    return x, chars, casing, y, index

We build the mapping on the training set only, then preprocess the three sets.
We also create the datasets and the dataloaders: each sample is now made of $5$ elements (GloVe embeddings, characters, binary features, label, and index of the token of interest).

In [ ]:
from torch.utils.data import Dataset, DataLoader


char_mapping = build_char_mapping(train)
print(f"Number of characters: {len(char_mapping)}")

x_train, chars_train, casing_train, y_train, index_train = preprocess_dataset_char(train, embeddings, char_mapping)
x_val, chars_val, casing_val, y_val, index_val = preprocess_dataset_char(validation, embeddings, char_mapping)
x_test, chars_test, casing_test, y_test, index_test = preprocess_dataset_char(test, embeddings, char_mapping)


class CharDataset(Dataset):
    def __init__(self, x, chars, casing, y, index):
        self.x = x
        self.chars = chars
        self.casing = casing
        self.y = y
        self.index = index

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.chars[idx], self.casing[idx], self.y[idx], self.index[idx]


dataloader_char_train = DataLoader(
    CharDataset(x_train, chars_train, casing_train, y_train, index_train),
    batch_size=64, shuffle=True, collate_fn=lambda x: x
)

dataloader_char_val = DataLoader(
    CharDataset(x_val, chars_val, casing_val, y_val, index_val),
    batch_size=64, shuffle=False, collate_fn=lambda x: x
)

dataloader_char_test = DataLoader(
    CharDataset(x_test, chars_test, casing_test, y_test, index_test),
    batch_size=64, shuffle=False, collate_fn=lambda x: x
)

### Question 10

We will build a recurrent neural network with the following architecture:

1. **Character-level representation of each token**:
    * an embedding layer for the characters ([`torch.nn.Embedding()`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)) with vectors of dimension $16$ (use `padding_idx=0`);
    * a 1D convolutional layer ([`torch.nn.Conv1d()`](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html)) with $50$ filters of size $3$ (use `padding=1`);
    * a max pooling over the characters of the token (ignoring the padding characters), followed by the ReLU activation function.
2. **Concatenation** of the GloVe embedding ($50$), the character-level representation ($50$), and the binary features ($4$), i.e., a vector of dimension $104$ for each token.
3. **Dropout** with probability $0.3$.
4. **Two bidirectional LSTM layers** with $128$ hidden variables and a dropout of $0.3$ between the two layers.
5. **Dropout** with probability $0.3$ on the hidden state of the token of interest.
6. **Linear layer** with $1$ output variable.

A few important notes:

* [`torch.nn.Conv1d()`](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html) expects inputs of shape `(batch_size, n_channels, length)`: here, the "batch" is made of all the tokens and the channels are the dimensions of the character embeddings. Use the `transpose()` method.
* To compute all the character-level representations at once, you can concatenate the tensors of all the sentences of the batch with `torch.concat()`, then split the result back into sentences with `torch.split()`.
* To ignore the padding characters in the max pooling, you can replace the corresponding values by $-\infty$ with the `masked_fill()` method before computing the maximum.

Complete the missing code in the `__init__()` and `forward()` methods of the `CharRecurrentNeuralNetwork()` class.

In [ ]:
class CharRecurrentNeuralNetwork(L.LightningModule):
    """Recurrent neural network using the characters of the tokens.

    Parameters
    ----------
    n_chars : int
        Number of characters (including padding and unknown characters).

    char_embedding_dim : int (default = 16)
        Dimension of the character embeddings.

    n_filters : int (default = 50)
        Number of filters of the convolutional layer.

    hidden_size : int (default = 128)
        Number of variables in the hidden state of the LSTM layers.

    dropout : float (default = 0.3)
        Dropout rate.
    """

    def __init__(self, n_chars, char_embedding_dim=16, n_filters=50, hidden_size=128, dropout=0.3):
        super().__init__()

        ### BEGIN TODO ###
        # self.char_embedding =
        # self.conv =
        # self.lstm =
        # self.dropout =
        # self.linear =
        #### END TODO ####

        self.loss = torch.nn.BCEWithLogitsLoss()
        self.accuracy_train = Accuracy(task="binary")
        self.accuracy_val = Accuracy(task="binary")
        self.accuracy_test = Accuracy(task="binary")

    def forward(self, x, chars, casing, index):
        """Implements the forward pass.

        Parameters
        ----------
        x : list[Tensor], len = batch_size
            GloVe embeddings of the tokens of each sentence.

        chars : list[Tensor], len = batch_size
            Characters of the tokens of each sentence.

        casing : list[Tensor], len = batch_size
            Binary features of the tokens of each sentence.

        index : tensor, shape = (batch_size,)
            Index of the token of interest in each sequence.

        Returns
        -------
        logits : Tensor, shape = (batch_size,)
            Logits.
        """
        ### BEGIN TODO ###
        # Compute the length of each sequence in this batch

        # Character-level representation of all the tokens of the batch

        # Concatenate the three representations and apply dropout

        # Split into sentences, pad and pack the sequences

        # Apply the LSTM layers and unpack the output

        # Get the hidden state of the token of interest

        # Apply dropout and the linear layer
        # logits =
        #### END TODO ####

        return logits

    def step(self, batch, dataset):
        """Perform a step.

        A step consists in mapping a batch of samples to the loss function for this batch of input samples.

        Parameters
        ----------
        batch : tuple
            A batch of samples.

        dataset : {"training", "validation", "test"}
            Data set used.

        Returns
        -------
        loss : Tensor, shape = (1,)
            Loss function evaluated for this batch of samples.
        """
        # Get the inputs, outputs, and indices for this batch
        x = [item[0].to(self.device) for item in batch]
        chars = [item[1].to(self.device) for item in batch]
        casing = [item[2].to(self.device) for item in batch]
        y = torch.tensor([item[3] for item in batch], device=self.device)
        index = torch.tensor([item[4] for item in batch], device=self.device)

        logits = self(x, chars, casing, index)  # Forward pass outputting the logits
        loss = self.loss(logits, y)  # Evaluation of the loss function
        y_pred = (logits > 0).to(torch.float32)  # Predictions of the model

        if dataset == "training":
            metric = self.accuracy_train
            name = "train"
            bar_step = True
        elif dataset == "validation":
            metric = self.accuracy_val
            name = "val"
            bar_step = False
        else:
            metric = self.accuracy_test
            name = "test"
            bar_step = False

        acc = metric(y_pred, y)
        self.log(f"loss_{name}", loss, batch_size=len(y), prog_bar=bar_step, on_step=bar_step, on_epoch=True)
        self.log(f"accuracy_{name}", acc, batch_size=len(y), prog_bar=bar_step, on_step=bar_step, on_epoch=True)

        return loss

    def training_step(self, batch):
        """Perform a training step."""
        return self.step(batch, "training")

    def validation_step(self, batch):
        """Perform a validation step."""
        return self.step(batch, "validation")

    def test_step(self, batch):
        """Perform a test step."""
        return self.step(batch, "test")

    def on_train_start(self):
        """Code to run at the beginning of training."""
        string = f"Version {self.trainer.logger.version}"
        print(f"{string}\n{'=' * len(string)}\n")

    def on_train_epoch_end(self):
        """Code to run at the end of each training epoch."""
        metrics = self.trainer.callback_metrics
        string = (f"""
            Epoch {self.trainer.current_epoch + 1} / {self.trainer.max_epochs}
            -----------------------------------------
            |  Data set  | Loss function | Accuracy |
            | ---------- | ------------- | -------- |
            |  Training  |{metrics['loss_train'].item():^15.5f}|{metrics['accuracy_train'].item():^10.4%}|
            | Validation |{metrics['loss_val'].item():^15.5f}|{metrics['accuracy_val'].item():^10.4%}|
            -----------------------------------------
        """)
        string = "\n".join([line.strip() for line in string.strip().split("\n")])
        print(string, "\n")

    def configure_optimizers(self):
        """Configure the optimization algorithms."""
        optimizer = torch.optim.Adam(self.parameters())
        return optimizer

### Question 11

Train a `CharRecurrentNeuralNetwork` model for $10$ epochs, then evaluate it on the validation and test sets.
Compare its performance with the performance of the previous models.

In [ ]:
# TODO